## Notebook42

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install umap-learn --quiet
! python -m spacy download en_core_web_sm

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c
import numpy as np

import funs
from funs import DSText, umap_text

import spacy

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
docs = (
    pl.read_parquet(ub + "data/agnews_pca.parquet")
    .filter(c.index == "test")
    .select(c.id, c.label, c.text)
    .rename({"id": "doc_id"})
    .sort(c.doc_id)
    .group_by(c.label, maintain_order=True)
    .head(50)
)

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
anno = DSText.process(docs.select(c.doc_id, c.text), nlp)

### Questions

`docs` and `anno` are the same 200 AG News articles from the last notebook,
50 each labeled `"Business"`, `"Sci/Tech"`, `"Sports"`, or `"World"`. That
notebook counted words within each article one at a time; this one turns
each article into a single point and asks how those points relate to one
another. Print results unless a question says otherwise.

1. Compute the TF-IDF table with `DSText.compute_tfidf(anno, min_df=0.01,
max_df=1.0)`, the same call from the last notebook. Print its shape and the
number of distinct lemmas it covers.

2. In the first block, pick two lemmas, `"olympic"` and `"profit"`, pivot
their TF-IDF scores into two columns, filling missing combinations with zero,
join `label` on, and plot every document as its `doc_id`, positioned by these
two scores and colored by `label`. In the second block, filter to the
documents with a nonzero score for both terms. How many are there? Compare
this to the book's `poem`/`novel` example, where authors who write in both
forms show up in the upper-right corner of the plot.

Zero. Eleven documents (all `Sports`) sit along the `olympic` axis with a
`profit` score of exactly 0, and eleven more (all `Business`) sit along the
`profit` axis with an `olympic` score of exactly 0; the other 178 sit at the
origin, using neither word. There is no upper-right corner here at all. The
book's authors write in mixed proportions of poetry and prose, so some of
them genuinely earn a spot away from both axes. `olympic` and `profit`
belong to two unrelated news beats instead of two overlapping literary
forms, so no article in this corpus has a reason to use both, and the
quadrant plot ends up as clean as it possibly could be.

3. The TF-IDF matrix from question 1 has a row for every document and a
column for every lemma. Pivot it into that wide shape, and report what
percentage of the entries are exactly zero, along with the average number of
nonzero terms per document out of the total vocabulary size.

97.4% of the matrix is zero. Every one of these 200 short wire articles uses,
on average, only about 24 of the corpus's 917 lemmas, a concrete number
behind the chapter's description of a TF-IDF matrix as a *sparse embedding*.

4. In the first block, build a 2D UMAP projection of the corpus with
`umap_text`, using `min_df=0.01` and `max_df=0.5` as the chapter does, and
plot each document's `label` at its projected position, colored by `label`.
In the second block, find each point's nearest neighbor in the projection and
print the share whose neighbor carries the same `label`, both overall and
broken down by `label`. Does the plot show four clean, separated clusters the
way the book's biography pages did?

No. The four labels form loose, overlapping neighborhoods rather than four
separated blobs; a `Sports` patch in the upper left sits right next to a
`World` patch, and `Business` and `Sci/Tech` interleave through the middle of
the plot. Measuring it directly confirms the impression: the nearest
neighbor of a point shares its label 50% of the time overall (100 of 200),
twice the 25% we would expect from random guessing among four labels, but
nowhere near the clean separation the chapter's biography pages achieved.
`Sports` is the easiest to keep together (60%) and `Sci/Tech` the hardest
(42%), a gap that resurfaces below when the same question is asked with the
full-dimensional distances instead of a 2D projection.

5. Compute the pairwise angle distance between every pair of documents with
`DSText.compute_distances(anno, min_df=0.01, max_df=0.5)`. Print its shape,
and confirm the row count matches $\binom{200}{2}$, the number of ways to
pick an unordered pair from 200 documents.

6. Each pair in `text_dist` is stored once, in `doc_id < doc_id2` order.
Stack the table with a copy of itself with the two ID columns swapped, so
that every document appears in the `doc_id` column with a row for every
other document.

7. The last notebook's `doc120027` was Michael Phelps's "Giddy Phelps
Touches Gold" article. Find its five closest matches in `text_dist_both`.

The closest match, `doc120159` at distance 1.04, is itself titled "Phelps
Eyes Fourth Gold," a second wire article about the same swimmer during the
same Olympics. It sits noticeably closer than the next four matches (all
above 1.30), which are Olympic articles about other athletes and events.

8. Find the single nearest neighbor of every document at once, and report
what proportion of documents have a nearest neighbor sharing their own
`label`, both overall and broken down by `label`.

70% of documents (140 of 200) have a same-topic nearest neighbor under the
full-dimensional angle distance, well above the 50% question 4 found in the
2D UMAP projection, since collapsing hundreds of dimensions down to two
necessarily throws away some of the structure that separates the topics.
The ranking across labels matches question 4's ranking exactly, though:
`Sports` is easiest to match correctly (80%, 40 of 50) and `Sci/Tech` is
hardest (56%, 28 of 50) in both the full-dimensional distances and the 2D
projection built from them.

9. The chapter argues that plain Euclidean distance is a poor choice for
document vectors because document length distorts it. Test this directly. In
the first block, build the raw (non-length-normalized) TF-IDF matrix with the
same `min_df`/`max_df` used above, compute the Euclidean distance between
every pair of rows, find each document's Euclidean nearest neighbor, and count
how often each document gets chosen as one. In the second block, print
`doc120070`'s Euclidean nearest neighbor and its angle-distance nearest
neighbor. Which document (or documents) get picked as the nearest neighbor
unreasonably often?

Two documents, `doc120007` and `doc120006`, are picked as the Euclidean
nearest neighbor for 70 and 66 of the 200 documents respectively. That means
136 documents (68% of the corpus) get one of just two articles as their
"closest" match. Both are long: 95 and 137 alphabetic tokens, against a
corpus average of 39. Their raw (non-normalized) TF-IDF vectors simply carry
more total mass than a typical short wire article's, which pulls them
closer to almost everything in Euclidean terms regardless of topic. For
example, `doc120070` (`World`, about a police case) gets `doc120007` (a
`Sci/Tech` article about a computer virus, and incidentally the same article
with the backslash line-wrap artifact from the last notebook's question 5)
as its Euclidean nearest neighbor, while the angle distance used in the rest
of this notebook correctly matches it to `doc120033`, another `World`
article about a different Vancouver police case:

10. In the first block, find the single closest pair of documents in the
whole corpus under the angle distance. In the second block, print both
articles' full text to confirm by eye what makes them so close.

At a distance of 0.37 (by far the smallest in the corpus, well under the
0.85 next-closest pair), `doc120195` and `doc120199` turn out to be two
independent wire versions of the exact same story, filed by Reuters minutes
apart: Medtronic's quarterly earnings. The angle distance has found a real
near-duplicate, not a coincidence of vocabulary.

11. `text_dist_both` also contains pairs at the maximum possible distance,
$\pi/2$, meaning the two documents share no filtered vocabulary at all. In
the first block, find the document involved in the most such pairs. In the
second block, print that document's text and look at its top TF-IDF terms to
explain why.

`doc120178` shares no filtered vocabulary at all with 162 of the other 199
documents, over 80% of the corpus. It is not a broken row: it is a short
article (16 surviving terms) about the cyclist Leontien Zijlaard-van
Moorsel, and spaCy's tokenizer splits her name into three separate tokens
(`Zijlaard`, `van`, `Moorsel`), each appearing in only one other document in
the whole corpus. Those three rare tokens get the highest possible IDF
weight and dominate the document's normalized direction, so unless another
article happens to mention this specific cyclist by name, the angle between
the two vectors lands at the maximum. A short document built around one
uncommon proper name is, geometrically, an outlier almost by construction.
That is worth remembering before reading too much into any single
document's distance to the rest of a corpus.